# Sanity check the created datasets
- verify that val is not polluted

In [3]:
from constraints.lightning_wrappers.modules import ProjectLightning
from constraints.datatools.datasets import CachedArtificalDataset
from constraints import get_experiment_folder, get_data_folder, show_torch_image
from constraints.transforms.transformers import RigidTransformer
from constraints.computers.loss_computers import ProjectLossComputer
from constraints.losses import OneSideSDFSquare
from constraints.models.affine import ProjectWithTemplateA 
from constraints.computers.loss_computers import CrossEntrAndOneSide

from pathlib import Path

import torch
import pytorch_lightning as pl
FODLER = get_experiment_folder(Path("ex3")/"project_debug")
DATA =get_data_folder() / "artificial" / "downloaded"
TRN_FOLDER_AFFINE = DATA / "trn" / "affine"
VAL_FOLDER_AFFINE = DATA / "val" / "affine"
TRN_FOLDER_DEFORM = DATA / "trn" / "deformed"
VAL_FOLDER_DEFORM = DATA / "val" / "deformed"

In [5]:
trn_dataset_affine = CachedArtificalDataset(TRN_FOLDER_AFFINE, sdf_mode="scipy",return_transform=True)
val_dataset_affine = CachedArtificalDataset(VAL_FOLDER_AFFINE, sdf_mode="scipy",return_transform=True)
trn_dataset_deform = CachedArtificalDataset(TRN_FOLDER_DEFORM, sdf_mode="scipy",return_transform=True)
val_dataset_deform = CachedArtificalDataset(VAL_FOLDER_DEFORM, sdf_mode="scipy",return_transform=True)

In [6]:
transform1 = trn_dataset_affine[0]["transform"]
transform2 = trn_dataset_deform[0]["transform"]

print("Affine transform: ", transform1.shape)
print("Deform transform: ", transform2.shape)

Affine transform:  torch.Size([2, 3])
Deform transform:  torch.Size([2, 256, 256])


In [8]:
# Check whether train/val splits share examples based on the `transform` field.
# We do an all-vs-all comparison and report any duplicate pairs.

def _to_2d_tensor(x):
    t = torch.as_tensor(x)
    if t.ndim == 0:
        t = t.unsqueeze(0)
    return t.reshape(1, -1).float()


def _collect_transforms(dataset):
    rows = []
    for i in range(len(dataset)):
        tr = dataset[i]["mask"]
        rows.append(_to_2d_tensor(tr))
    if not rows:
        return torch.empty((0, 0), dtype=torch.float32)
    return torch.cat(rows, dim=0)


def compare_train_val_by_transform(trn_dataset, val_dataset, name, atol=1e-7, rtol=1e-6):
    trn = _collect_transforms(trn_dataset)
    val = _collect_transforms(val_dataset)

    if trn.numel() == 0 or val.numel() == 0:
        print(f"[{name}] Empty split detected. trn={len(trn_dataset)}, val={len(val_dataset)}")
        return []

    if trn.shape[1] != val.shape[1]:
        raise ValueError(
            f"[{name}] Transform dimensionality mismatch: trn dim={trn.shape[1]} vs val dim={val.shape[1]}"
        )

    duplicates = []
    for i in range(trn.shape[0]):
        # compare trn[i] against all val transforms at once
        close_mask = torch.isclose(trn[i].unsqueeze(0), val, atol=atol, rtol=rtol).all(dim=1)
        match_js = torch.where(close_mask)[0].tolist()
        for j in match_js:
            duplicates.append((i, j))

    total_pairs = trn.shape[0] * val.shape[0]
    print(f"\n[{name}] trn={trn.shape[0]}, val={val.shape[0]}, compared_pairs={total_pairs}")
    print(f"[{name}] duplicate transform pairs found: {len(duplicates)}")

    if duplicates:
        print(f"[{name}] First duplicate index pairs (trn_idx, val_idx): {duplicates[:20]}")
    else:
        print(f"[{name}] No duplicate transforms across train/val.")

    return duplicates


# Run checks for both dataset variants
affine_duplicates = compare_train_val_by_transform(
    trn_dataset_affine, val_dataset_affine, name="AFFINE"
)

deform_duplicates = compare_train_val_by_transform(
    trn_dataset_deform, val_dataset_deform, name="DEFORM"
)

if not affine_duplicates and not deform_duplicates:
    print("\nResult: train/val splits look clean based on transforms.")
else:
    print("\nResult: potential train/val leakage detected (see duplicate index pairs above).")


[AFFINE] trn=2000, val=100, compared_pairs=200000
[AFFINE] duplicate transform pairs found: 0
[AFFINE] No duplicate transforms across train/val.

[DEFORM] trn=2000, val=100, compared_pairs=200000
[DEFORM] duplicate transform pairs found: 0
[DEFORM] No duplicate transforms across train/val.

Result: train/val splits look clean based on transforms.


# Sanity check UNET


In [3]:

import torch
import segmentation_models_pytorch as smp
unet = smp.Unet(
            "resnet18", encoder_weights="imagenet", in_channels=1, classes=3
        )

In [4]:
logits = unet(torch.rand(1, 1, 256, 256))

In [5]:
print(logits.max(), logits.min(), logits.shape)

tensor(6.0988, grad_fn=<MaxBackward1>) tensor(-3.8231, grad_fn=<MinBackward1>) torch.Size([1, 3, 256, 256])
